# 03 · Análisis exploratorio de datos financieros

**Financial Analytics Lab · Python · pandas · NumPy · matplotlib**

**Pregunta orientadora:** ¿qué revelan los rendimientos históricos sobre el comportamiento y la variabilidad de los activos de un portafolio?

En los módulos anteriores aprendimos a representar inversiones y preparar precios históricos. Ahora analizaremos los rendimientos mediante medidas descriptivas y visualizaciones. El objetivo es **explicar qué representa cada indicador, verificar su cálculo e interpretar sus limitaciones**.

**Aprendizajes:** rendimiento simple; media y mediana; varianza y desviación estándar; cuantiles; distribuciones; precios normalizados; gráficos; interpretación financiera.

**Trabajo autónomo:** ejecuta las celdas en orden, completa cada «Tu práctica» y redacta conclusiones propias. Los datos provienen de un servicio externo y sus resultados pueden variar.

## 0 · Planteamiento del problema

Un analista necesita comparar tres activos: MSFT, AAPL y JPM. ¿Cuál presentó mayor rendimiento promedio por observación? ¿Cuál mostró mayor dispersión? ¿Qué revelan sus distribuciones?

Un precio nominal más alto no implica mayor rentabilidad. Para comparar activos utilizaremos variaciones relativas y medidas estadísticas.

**Alcance:** el análisis describe un período histórico; no permite anticipar rendimientos ni recomendar inversiones.

## 1 · Preparar los datos

En el módulo 02 generamos `precios_portafolio.csv`. Puedes utilizar ese archivo si lo conservaste. Para que el cuaderno también funcione por separado, la opción predeterminada consulta precios históricos mediante `yfinance`.

**ETL:** extracción de precios, validación de fechas y valores, y preparación de una tabla común. `auto_adjust=True` solicita precios ajustados según la fuente. El límite `end` es exclusivo.

In [ ]:
%pip -q install yfinance pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timezone

TICKERS = ["MSFT", "AAPL", "JPM"]
INICIO = "2024-01-01"
FIN = "2025-01-01"
USAR_CSV_MODULO_02 = False
print("Fecha de consulta UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

In [ ]:
if USAR_CSV_MODULO_02:
    from google.colab import files
    archivos = files.upload()
    if "precios_portafolio.csv" not in archivos:
        raise FileNotFoundError("Sube precios_portafolio.csv del módulo 02.")
    precios = pd.read_csv("precios_portafolio.csv", index_col="Fecha", parse_dates=["Fecha"])
else:
    series = []
    for ticker in TICKERS:
        tabla = yf.download(ticker, start=INICIO, end=FIN,
                            auto_adjust=True, progress=False, multi_level_index=False)
        if tabla.empty or "Close" not in tabla.columns:
            raise RuntimeError(f"No se pudo consultar {ticker}. Verifica la conexión y la fuente.")
        series.append(tabla[["Close"]].rename(columns={"Close": ticker}))
    precios = pd.concat(series, axis=1, join="inner")

precios.index = pd.to_datetime(precios.index)
precios = precios.sort_index()
precios = precios.loc[~precios.index.duplicated(keep="first")]
precios = precios.apply(pd.to_numeric, errors="coerce").dropna(how="any")
precios = precios.loc[(precios > 0).all(axis=1)]
if len(precios) < 3:
    raise RuntimeError("No hay suficientes observaciones válidas.")
print("Dimensiones:", precios.shape)
display(precios.head())
print("Valores faltantes:", precios.isna().sum().to_dict())

**Tu práctica 1:** identifica la primera y última fecha, número de observaciones y cantidad de activos. Explica por qué es necesario comparar fechas comunes y por qué no se deben inventar precios para completar observaciones faltantes.

In [ ]:
# TU PRÁCTICA 1
# Explora las fechas y documenta el diagnóstico de calidad.

## 2 · Rendimientos simples

El rendimiento entre dos observaciones consecutivas se define como:

\[r_t = \frac{P_t}{P_{t-1}}-1\]

Un resultado de 0,02 representa un rendimiento del 2 %. El primer rendimiento no está definido porque falta un precio anterior. `pct_change(fill_method=None)` calcula las variaciones relativas de cada columna.

In [ ]:
rendimientos = precios.pct_change(fill_method=None).dropna(how="any")
rendimientos_pct = rendimientos * 100
display(rendimientos_pct.head())

activo = precios.columns[0]
manual = precios[activo].iloc[1] / precios[activo].iloc[0] - 1
calculado = rendimientos[activo].iloc[0]
assert np.isclose(manual, calculado)
print(f"Comprobación {activo}: {manual:.6f} = {calculado:.6f}")

**Tu práctica 2:** comprueba manualmente el rendimiento de otro activo y explica el signo del resultado. Distingue entre precio (USD por acción) y rendimiento (proporción o porcentaje).

In [ ]:
# TU PRÁCTICA 2
# Verifica un rendimiento de otro activo con la fórmula.

## 3 · Media y mediana

La **media aritmética** resume el rendimiento promedio por observación: \(\bar r = \sum r_i/n\). La **mediana** divide los valores ordenados en dos mitades y suele ser menos sensible a valores extremos.

**Importante:** el promedio de rendimientos por observación no equivale al rendimiento acumulado del período ni a una rentabilidad anual.

In [ ]:
centro = pd.DataFrame({
    "Media (%)": rendimientos_pct.mean(),
    "Mediana (%)": rendimientos_pct.median()
})
display(centro.round(4))
muestra = rendimientos_pct.iloc[:5, 0]
media_manual = sum(muestra) / len(muestra)
assert np.isclose(media_manual, muestra.mean())
print("Media de cinco observaciones comprobada:", round(media_manual, 5))

**Tu práctica 3:** compara la media y la mediana de cada activo. ¿En cuál es mayor su diferencia? Propón una explicación que puedas contrastar después con un histograma.

In [ ]:
# TU PRÁCTICA 3
# Calcula las diferencias absolutas y redacta una interpretación.

## 4 · Dispersión y volatilidad histórica

La **varianza muestral** resume las desviaciones cuadráticas respecto a la media: \(s^2=\sum(r_i-\bar r)^2/(n-1)\). La **desviación estándar** es su raíz cuadrada. Utilizaremos `ddof=1` para trabajar con la definición muestral.

En series de rendimientos, la desviación estándar es una medida habitual de volatilidad histórica. Sin embargo, no captura todas las dimensiones del riesgo financiero.

In [ ]:
dispersion = pd.DataFrame({
    "Desv. estándar (%)": rendimientos_pct.std(ddof=1),
    "Mínimo (%)": rendimientos_pct.min(),
    "Máximo (%)": rendimientos_pct.max()
})
display(dispersion.round(4))
muestra = rendimientos_pct.iloc[:5, 0].to_numpy()
media = muestra.mean()
varianza_manual = ((muestra - media) ** 2).sum() / (len(muestra) - 1)
desviacion_manual = np.sqrt(varianza_manual)
assert np.isclose(desviacion_manual, pd.Series(muestra).std(ddof=1))
print("Desviación estándar comprobada:", round(desviacion_manual, 5))

**Tu práctica 4:** ordena los activos por desviación estándar. Explica por qué una desviación mayor describe más dispersión en esta muestra, pero no demuestra por sí sola que el activo sea una peor inversión.

In [ ]:
# TU PRÁCTICA 4
# Ordena la tabla y explica la comparación.

## 5 · Cuantiles y valores extremos

Los **cuantiles** dividen la distribución ordenada. Q1 es el percentil 25 y Q3 el percentil 75. El **rango intercuartílico** es Q3 − Q1.

Una regla exploratoria identifica observaciones fuera de Q1 − 1,5 × IQR y Q3 + 1,5 × IQR. Esos valores requieren interpretación, no eliminación automática: movimientos extremos pueden corresponder a acontecimientos reales.

In [ ]:
q1 = rendimientos_pct.quantile(0.25)
q3 = rendimientos_pct.quantile(0.75)
iqr = q3 - q1
fuera = rendimientos_pct.lt(q1 - 1.5 * iqr) | rendimientos_pct.gt(q3 + 1.5 * iqr)
cuantiles = pd.DataFrame({
    "Q1 (%)": q1, "Q3 (%)": q3, "IQR (pp)": iqr,
    "Observaciones fuera de límites": fuera.sum()
})
display(cuantiles.round(4))

**Tu práctica 5:** identifica el activo con más observaciones fuera de los límites exploratorios. Explica por qué no debes eliminarlas sin verificar su origen.

In [ ]:
# TU PRÁCTICA 5
# Examina cuantiles y valores extremos.

## 6 · Visualizar los resultados

Un gráfico debe responder una pregunta:

- **Precios normalizados (base 100):** ¿cómo evolucionó cada activo respecto a su nivel inicial?
- **Histograma:** ¿cómo se distribuyeron los rendimientos?
- **Diagrama de caja:** ¿cómo se comparan medianas, cuartiles y observaciones extremas?

Los precios normalizados se calculan como 100 × precio actual / precio inicial. No incluyen dividendos ni otros flujos adicionales.

In [ ]:
normalizados = precios.div(precios.iloc[0]).mul(100)
ax = normalizados.plot(figsize=(10, 4))
ax.set(title="Precios normalizados (base 100)", xlabel="Fecha", ylabel="Índice")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for ticker in rendimientos_pct.columns:
    ax.hist(rendimientos_pct[ticker], bins=35, alpha=0.4, label=ticker)
ax.set(title="Distribución de rendimientos", xlabel="Rendimiento (%)", ylabel="Frecuencia")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([rendimientos_pct[t].to_numpy() for t in rendimientos_pct.columns],
           labels=list(rendimientos_pct.columns), showmeans=True)
ax.set(title="Dispersión de rendimientos", ylabel="Rendimiento (%)")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretación guiada:** los precios normalizados muestran variaciones relativas acumuladas del nivel de cotización. El histograma permite observar concentración y colas. El diagrama de caja facilita comparar medianas y dispersión central. La forma del histograma, por sí sola, no prueba que una distribución sea normal.

**Tu práctica 6:** redacta una interpretación para cada gráfico. Diferencia explícitamente lo observado de las conclusiones que no pueden sostenerse.

In [ ]:
# TU PRÁCTICA 6
# Puedes crear un gráfico individual del activo que te interese.

## 7 · Integrar los indicadores en un informe

Una conclusión rigurosa conecta varios resultados. Un activo puede presentar una media histórica más alta y, al mismo tiempo, una dispersión mayor. No existe una clasificación automática de «mejor activo» sin conocer objetivos y restricciones de inversión.

Distingue siempre **descripción** (qué ocurrió), **interpretación** (qué significa) y **limitación** (qué no podemos concluir).

In [ ]:
resumen = pd.DataFrame({
    "Media (%)": rendimientos_pct.mean(),
    "Mediana (%)": rendimientos_pct.median(),
    "Desv. estándar (%)": rendimientos_pct.std(ddof=1),
    "Percentil 5 (%)": rendimientos_pct.quantile(0.05),
    "Percentil 95 (%)": rendimientos_pct.quantile(0.95),
    "Mínimo (%)": rendimientos_pct.min(),
    "Máximo (%)": rendimientos_pct.max(),
    "N": rendimientos_pct.count()
})
display(resumen.round(4))
resumen.to_csv("resumen_exploratorio_portafolio.csv", index_label="Ticker")
print("Archivo creado: resumen_exploratorio_portafolio.csv")

**Tu práctica 7:** redacta entre 100 y 150 palabras comparando los activos mediante media, mediana, dispersión y al menos un gráfico. Sustenta cada afirmación en una medida calculada.

## 8 · Reto integrador: informe exploratorio de un portafolio

Desarrolla un análisis original con tres activos y un período explícito. Debe incluir:

1. Pregunta de análisis, selección de activos y fuente.
2. Diagnóstico de fechas, tipos, datos faltantes y alineación.
3. Rendimientos simples y comprobación manual de una observación.
4. Media, mediana, desviación estándar, mínimos, máximos y cuartiles.
5. Comprobación manual de la media y desviación estándar de cinco rendimientos.
6. Gráficos de precios normalizados, histograma y diagrama de caja.
7. Interpretación comparativa de **180 a 250 palabras**, con supuestos y limitaciones.
8. Exportación del resumen estadístico a CSV.

**Evidencia:** notebook ejecutable con código, cálculos, gráficos rotulados y conclusiones. Documenta el uso de recursos externos o herramientas de IA. No inventes resultados si falla la consulta.

In [ ]:
# RETO INTEGRADOR — SOLUCIÓN PROPIA
# Añade aquí tu consulta, cálculos, visualizaciones y comprobaciones.

## 9 · Autoevaluación y continuidad

- [ ] Distingo precios de rendimientos.
- [ ] Verifico manualmente un rendimiento simple.
- [ ] Interpreto media y mediana sin confundirlas con rendimiento acumulado.
- [ ] Comprendo la desviación estándar muestral y sus limitaciones.
- [ ] Examino cuantiles y observaciones extremas.
- [ ] Utilizo gráficos adecuados y explico sus resultados.
- [ ] Documento las decisiones metodológicas y la fuente.

**Próximo módulo:** series temporales financieras: frecuencia, tendencia, medias móviles, volatilidad móvil y rendimientos acumulados.

**Referencias:** [pandas](https://pandas.pydata.org/docs/) · [NumPy](https://numpy.org/doc/stable/) · [matplotlib](https://matplotlib.org/stable/) · [yfinance](https://ranaroussi.github.io/yfinance/).

Los resultados históricos son descriptivos y no constituyen recomendaciones de inversión.

## Quiz de cierre · Análisis exploratorio

**Quiz evaluativo 3 de 4 · componente de quices (15 % total)**

Responde individualmente las cinco preguntas. Las respuestas deben ser originales, razonadas y coherentes con el código que ejecutes. No basta con escribir una definición: explica el significado de los resultados cuando se solicite.

**Criterio sugerido:** cada pregunta vale 1 punto (5 puntos en total). Las preguntas 1–2 evalúan comprensión, 3–4 interpretación y 5 aplicación. La docente verificará las respuestas y registrará la nota conforme a la ponderación institucional de cada quiz.

### Pregunta 1 · Comprensión

¿Por qué no es correcto comparar directamente precios nominales para determinar cuál activo fue más rentable?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 2 · Comprensión

Explica qué informa la desviación estándar de los rendimientos.

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 3 · Interpretación

Un activo tiene rendimiento medio diario positivo y volatilidad alta. ¿Qué podemos concluir y qué no?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 4 · Interpretación

¿Cómo interpretarías una correlación de -0.7 entre rendimientos de dos activos?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 5 · Aplicación

Crea una serie de al menos cinco rendimientos diarios, calcula media y desviación estándar con pandas y escribe una interpretación.

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._



In [ ]:
# Pregunta 5 — solución individual
# Escribe y ejecuta aquí tu código.
